# Track B — Corrected Ablation Batch
### Physical-reasoning distribution in Gemma-2 2B

This notebook reruns the six-prompt ablation curve using the **validated** intervention method (scaling each feature's own activation to zero), fixing the earlier batch that used raw constants and produced garbage tokens.

It **loads the saved graphs from Drive** so you do not re-trace, and it **flags garbage-token flips** so a corrupted result cannot be mistaken for a real threshold.

**Before running:** Runtime > Change runtime type > GPU. You need the same HF access and Drive backup (`track_b_graphs/*.pt`) as before.

Run top to bottom. The only cell that needs your input is the token login.

## 1  GPU check

In [ ]:
import subprocess, torch
try:
    print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader']).decode().strip())
except Exception as e:
    print('No GPU - Runtime > Change runtime type > GPU. ', e)
print('CUDA:', torch.cuda.is_available())

Tesla T4, 15360 MiB
CUDA: True


## 2  Install circuit-tracer

In [ ]:
import os, subprocess
if not os.path.isdir('circuit-tracer'):
    subprocess.run(['git','clone','https://github.com/safety-research/circuit-tracer.git'], check=True)
%pip install -q ./circuit-tracer
print('install done - if a dependency conflict is reported it is harmless; verify with the import cell next')

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 504.6 kB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.0/296.0 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 968.6/968.6 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 102.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.2/110.2 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

## 3  Authenticate + confirm Gemma access

In [ ]:
from huggingface_hub import login, whoami, HfApi
login()  # paste a Read token (or fine-grained with gated-repo read access)
print('Logged in as:', whoami()['name'])
try:
    HfApi().model_info('google/gemma-2-2b'); print('Access to google/gemma-2-2b: GRANTED')
except Exception as e:
    print('Access DENIED - accept licence at https://huggingface.co/google/gemma-2-2b then re-run'); raise

Logged in as: Boozelightyear
Access to google/gemma-2-2b: GRANTED


## 4  Mount Drive (where the saved graphs live)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/track_b_graphs'
import os
present = [f for f in os.listdir(BASE) if f.endswith('.pt')] if os.path.isdir(BASE) else []
print('graph files found on Drive:', present)
assert present, 'No .pt graphs found. Check BASE path matches your Drive backup folder.'

Mounted at /content/drive
graph files found on Drive: ['float_sink.pt', 'support_remove.pt', 'tip_over.pt', 'inertia.pt', 'sanity_dallas.pt', 'melt.pt', 'collision.pt']


## 5  Load the model + transcoders

In [ ]:
import torch
from circuit_tracer import ReplacementModel, Graph
model = ReplacementModel.from_pretrained('google/gemma-2-2b', 'gemma', dtype=torch.bfloat16)
print('model + transcoders loaded')

config.yaml:   0%|          | 0.00/202 [00:00<?, ?B/s]

Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

layer_0.safetensors:   0%|          | 0.00/302M [00:00<?, ?B/s]

layer_12.safetensors:   0%|          | 0.00/302M [00:00<?, ?B/s]

layer_15.safetensors:   0%|          | 0.00/302M [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Loaded pretrained model google/gemma-2-2b into HookedTransformer
model + transcoders loaded


## 6  Validate the method first (Dallas -> Austin)
Do not trust any physics number until this flips to a **real word**. This is the positive control.

In [ ]:
import os, torch

BASE = '/content/drive/MyDrive/track_b_graphs'   # same folder as your Drive backup

def predicted_next(prompt):
    toks = model.tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        logits = model(toks['input_ids'])
    return model.tokenizer.decode([logits[0,-1].argmax().item()])

def rank(g, k):
    A = g.adjacency_matrix; n = g.active_features.shape[0]
    infl = A[-1,:n].abs(); vals, idx = torch.topk(infl, min(k,n))
    return [tuple(g.active_features[i].tolist()) for i in idx]

def is_garbage(tok):
    t = tok.strip()
    return (len(t)==0) or ('\r' in tok) or ('\n' in tok) or (not any(ch.isalpha() for ch in t))

def ablate_to_zero(prompt, feats, clean_acts):
    interv = [(l,p,f, clean_acts[l,p,f].item()*0.0) for (l,p,f) in feats]
    logits,_ = model.feature_intervention(prompt, interv)
    return model.tokenizer.decode([logits[0,-1].argmax().item()])

dallas = 'Fact: The capital of the state containing Dallas is'
dg = Graph.from_pt(f'{BASE}/sanity_dallas.pt') if os.path.exists(f'{BASE}/sanity_dallas.pt') else None
if dg is not None:
    clean = predicted_next(dallas)
    _, ca = model.feature_intervention(dallas, [])
    ranked = rank(dg, 50)
    print(f'Dallas clean -> {clean!r}')
    for k in (5,10,25,50):
        tok = ablate_to_zero(dallas, ranked[:k], ca)
        print(f'  top-{k:3d}: {tok!r}  {"FLIP" if tok!=clean else ""}  {"(garbage!)" if is_garbage(tok) else ""}')
    print('\nMethod is valid IF a flip above lands on a real word (e.g. the/not), not garbage.')
else:
    print('sanity_dallas.pt not on Drive - skipping validation. Consider re-tracing Dallas.')

Dallas clean -> ' Austin'
  top-  5: ' the'  FLIP  
  top- 10: ' the'  FLIP  
  top- 25: ' not'  FLIP  
  top- 50: ' not'  FLIP  

Method is valid IF a flip above lands on a real word (e.g. the/not), not garbage.


## 7  Six-prompt ablation batch (corrected)
Loads each graph from Drive one at a time. Any flip to a garbage token is flagged and NOT counted as a real threshold.

In [ ]:
import gc
PROMPTS = {
 'support_remove':'A cup rests on a table. If the table is removed, the cup will',
 'collision':'A fast ball hits a still ball head-on. The still ball starts to',
 'tip_over':'A full glass of water is knocked over. The water spills',
 'float_sink':'A heavy metal block is placed in water. It will',
 'melt':'An ice cube is left in a warm room. After an hour it will',
 'inertia':'A book slides across a rough table. Over time it will slow',
}
KS = (1,5,10,25,50,100,150,175,200,250,300)

def curve(name):
    g = Graph.from_pt(f'{BASE}/{name}.pt')
    prompt = PROMPTS[name]
    clean = predicted_next(prompt)
    _, ca = model.feature_intervention(prompt, [])
    ranked = rank(g, max(KS))
    print(f'\n=== {name} (clean -> {clean!r}) ===')
    real_flip = None
    for k in KS:
        if k > len(ranked): break
        tok = ablate_to_zero(prompt, ranked[:k], ca)
        changed = tok != clean
        garbage = is_garbage(tok)
        flag = '(GARBAGE - not a valid flip)' if (changed and garbage) else ('FLIP' if changed else '')
        print(f'  top-{k:3d}: {tok!r:12s} {flag}')
        if changed and not garbage and real_flip is None:
            real_flip = k
    print(f'  -> real-word flip threshold: {real_flip if real_flip else ">"+str(max(KS))}')
    del g; gc.collect(); torch.cuda.empty_cache()
    return real_flip

results = {}
for name in PROMPTS:
    try:
        results[name] = curve(name)
    except Exception as e:
        print(f'{name}: ERROR {type(e).__name__}: {e}')
        results[name] = None

print('\n=== SUMMARY: real-word flip thresholds ===')
for n,t in results.items():
    print(f'  {n:16s}: {t if t else ">300 or none"}')


=== support_remove (clean -> ' fall') ===
  top-  1: ' fall'      
  top-  5: ' fall'      
  top- 10: ' fall'      
  top- 25: ' fall'      
  top- 50: ' fall'      
  top-100: ' fall'      
  top-150: ' fall'      
  top-175: ' fall'      
  top-200: ' fall'      
  top-250: ' fall'      
  top-300: ' fall'      
  -> real-word flip threshold: >300

=== collision (clean -> ' move') ===
  top-  1: ' move'      
  top-  5: ' move'      
  top- 10: ' move'      
  top- 25: ' move'      
  top- 50: ' move'      
  top-100: ' move'      
  top-150: ' move'      
  top-175: ' move'      
  top-200: ' move'      
  top-250: ' move'      
  top-300: ' move'      
  -> real-word flip threshold: >300

=== tip_over (clean -> ' out') ===
  top-  1: ' out'       
  top-  5: ' out'       
  top- 10: ' out'       
  top- 25: ' out'       
  top- 50: ' out'       
  top-100: ' on'        FLIP
  top-150: ' out'       
  top-175: ' out'       
  top-200: ' out'       
  top-250: ' out'       
  top-3

## 8  Specificity control (random ablation)
Ablating the same number of RANDOM features should NOT flip the prediction. This is what makes the ranked-feature result meaningful.

In [ ]:
def random_control(name, k=100, seed=0):
    g = Graph.from_pt(f'{BASE}/{name}.pt')
    prompt = PROMPTS[name]
    clean = predicted_next(prompt)
    _, ca = model.feature_intervention(prompt, [])
    n = g.active_features.shape[0]
    torch.manual_seed(seed)
    idx = torch.randperm(n)[:k]
    feats = [tuple(g.active_features[i].tolist()) for i in idx]
    tok = ablate_to_zero(prompt, feats, ca)
    print(f'{name:16s}: clean {clean!r} | random-{k} -> {tok!r}  {"(survived - good)" if tok==clean else "(changed)"}')
    del g; gc.collect(); torch.cuda.empty_cache()

for name in PROMPTS:
    random_control(name)

support_remove  : clean ' fall' | random-100 -> ' fall'  (survived - good)
collision       : clean ' move' | random-100 -> ' move'  (survived - good)
tip_over        : clean ' out' | random-100 -> ' out'  (survived - good)
float_sink      : clean ' sink' | random-100 -> ' sink'  (survived - good)
melt            : clean ' melt' | random-100 -> ' melt'  (survived - good)
inertia         : clean ' down' | random-100 -> ' down'  (survived - good)


## 9  Save results to Drive
Run before disconnecting so the corrected numbers persist.

In [ ]:
summary = '\n'.join(f'{n}: {t}' for n,t in results.items())
with open(f'{BASE}/corrected_ablation_thresholds.txt','w') as f:
    f.write('Track B corrected ablation (real-word flip thresholds)\n')
    f.write(summary)
print('saved to Drive:')
print(summary)

saved to Drive:
support_remove: None
collision: None
tip_over: 100
float_sink: 200
melt: 5
inertia: 50


---
### How to read the output
- A threshold is only trustworthy if the flip lands on a **real word**. Any `(GARBAGE ...)` flag means that prompt's ablation is still corrupting the model and its number should NOT be used.
- float_sink is the known-good case; expect it to hold until ~175 and flip around 200 to a real word.
- Paste the Section 7 summary and the Section 8 control output back for the dissertation update.